# 🚕 Ćwiczenie 11: BigQuery i Dataproc — Przejazdy taksówek w Chicago

**Kurs:** Python w Zastosowaniach Big Data
**Zajęcia nr:** 11
**Szacowany czas:** ⏱️ ~90 minut
**Poziom trudności:** ⭐⭐⭐⭐ (Zaawansowany)

## 📚 Cele ćwiczenia

Po ukończeniu tego notebooka będziesz potrafić:
1. Odpytać publiczny dataset BigQuery (`bigquery-public-data.chicago_taxi_trips.taxi_trips`) — lub jego offline'owy odpowiednik
2. Przeanalizować wzorce w danych o przejazdach taksówek (godziny szczytu, napiwki, dystanse)
3. Oszacować **koszt** zapytania BigQuery na podstawie liczby przeskanowanych bajtów
4. Przełożyć logikę SQL na PySpark, tak jakby uruchamiać ją na klastrze Dataproc

## 📋 Wymagana wiedza wstępna
- Podstawy pandas (`groupby`, `agg`, `corr`)
- Podstawy SQL (`SELECT`, `GROUP BY`, `WHERE`)
- Wykład 11: BigQuery, Dataproc, koszty i partycjonowanie

---
> 💡 **Tip:** Uruchamiaj komórki po kolei (`Shift+Enter`). Jeśli utkniesz — sprawdź hinty lub zapytaj prowadzącego!

## 🛣️ Dwie ścieżki do wyboru

Ten notebook został zaprojektowany, by działał **zawsze**, niezależnie od tego, czy masz dostęp do konta Google Cloud:

### Ścieżka A — Prawdziwy BigQuery (opcjonalna, wymaga darmowego konta GCP) ☁️
Jeśli masz założony projekt GCP (darmowy tier wystarczy — 1 TB zapytań/msc za darmo), możesz uruchomić prawdziwe zapytania SQL na publicznym zbiorze danych `bigquery-public-data.chicago_taxi_trips.taxi_trips` (~200 mln wierszy!). W Google Colab wystarczy:

```python
from google.colab import auth
auth.authenticate_user()
%load_ext google.cloud.bigquery
```

```sql
%%bigquery wyniki --project TWOJ-PROJEKT-GCP
SELECT EXTRACT(HOUR FROM trip_start_timestamp) AS godzina, AVG(fare) AS srednia_stawka
FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
WHERE trip_start_timestamp BETWEEN '2023-01-01' AND '2023-01-31'
GROUP BY godzina ORDER BY godzina
```

Kod SQL powyżej jest pokazany jako **materiał referencyjny** — jeśli masz GCP, odkomentuj i uruchom!

### Ścieżka B — Offline fallback (domyślna, działa zawsze) 🧪
Generujemy **syntetyczny** DataFrame `taxi_df`, który naśladuje strukturę i statystyki prawdziwego datasetu Chicago Taxi Trips (godziny szczytu, napiwki, dystanse, dzielnice). Wszystkie ćwiczenia w tym notebooku wykonujemy na **Ścieżce B**, żeby każdy student — z kontem GCP czy bez — mógł je ukończyć.

> 🔑 Wszędzie tam, gdzie w komentarzach widzisz blok `-- SQL (Ścieżka A)`, masz gotowe zapytanie do przetestowania na prawdziwym BigQuery, jeśli masz taką możliwość.

In [ ]:
# @title 🔧 Setup — Run this cell first!

# Ścieżka A (opcjonalnie, wymaga konta GCP) — odkomentuj, jeśli chcesz uruchomić prawdziwy BigQuery:
# !pip install -q google-cloud-bigquery pandas-gbq
# from google.colab import auth
# auth.authenticate_user()

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 50)

np.random.seed(42)

print("✅ Setup complete! Ruszamy z analizą przejazdów taksówek 🚕")

## 📊 Generowanie danych — Ścieżka B (offline fallback)

Generujemy ~50 000 wierszy syntetycznych przejazdów taksówek, naśladujących strukturę `bigquery-public-data.chicago_taxi_trips.taxi_trips`:

| Kolumna | Opis |
|---------|------|
| `trip_id` | unikalny identyfikator przejazdu |
| `pickup_datetime` | data i godzina rozpoczęcia przejazdu (2023) |
| `pickup_community_area` | dzielnica odbioru pasażera (1-77, jak w Chicago) |
| `trip_miles` | dystans przejazdu w milach |
| `trip_seconds` | czas trwania przejazdu w sekundach |
| `fare` | stawka za przejazd (USD) |
| `tips` | napiwek (USD) |
| `payment_type` | `Credit Card` / `Cash` / `Mobile` / `Other` |
| `company` | firma taksówkowa |

In [ ]:
# @title 📊 Generowanie syntetycznego zbioru danych (Ścieżka B)

N = 50_000

# Popularne dzielnice odbioru mają większą wagę (np. centrum, lotnisko)
obszary = np.arange(1, 78)
wagi_obszarow = np.ones(77)
wagi_obszarow[[7, 31, 32, 76]] = 8.0  # popularne dzielnice (centrum, lotnisko O'Hare...)
wagi_obszarow /= wagi_obszarow.sum()

# Losowe daty w 2023 roku, z większym natężeniem w godzinach szczytu
start = pd.Timestamp("2023-01-01")
dni = np.random.randint(0, 365, size=N)
godziny_wagi = np.array([1, 1, 1, 1, 1, 2, 4, 6, 5, 4, 3, 3, 4, 4, 4, 4, 5, 6, 7, 6, 4, 3, 2, 1], dtype=float)
godziny_wagi /= godziny_wagi.sum()
godziny = np.random.choice(np.arange(24), size=N, p=godziny_wagi)
minuty = np.random.randint(0, 60, size=N)
pickup_datetime = start + pd.to_timedelta(dni, unit="D") + pd.to_timedelta(godziny, unit="h") + pd.to_timedelta(minuty, unit="m")

trip_miles = np.round(np.random.gamma(shape=2.0, scale=2.2, size=N), 2)
trip_seconds = np.round(trip_miles * np.random.normal(220, 40, size=N) + np.random.normal(60, 30, size=N)).clip(min=60).astype(int)

# Stawka zależy od dystansu i czasu (jak prawdziwy taksometr) + szum
fare = np.round(3.25 + trip_miles * 2.25 + trip_seconds / 60 * 0.35 + np.random.normal(0, 1.5, size=N), 2).clip(min=3.25)

payment_type = np.random.choice(
    ["Credit Card", "Cash", "Mobile", "Other"], size=N, p=[0.55, 0.30, 0.12, 0.03]
)

# Napiwki: karty i mobile dają wyższy % napiwku niż gotówka
procent_napiwku = np.where(
    np.isin(payment_type, ["Credit Card", "Mobile"]),
    np.random.normal(0.18, 0.05, size=N),
    np.random.normal(0.03, 0.03, size=N),
).clip(min=0)
tips = np.round(fare * procent_napiwku, 2)

company = np.random.choice(
    ["Flash Cab", "Yellow Cab", "Taxi Affiliation Services", "City Service", "Chicago Carriage Cab"],
    size=N,
    p=[0.30, 0.25, 0.20, 0.15, 0.10],
)

taxi_df = pd.DataFrame({
    "trip_id": np.arange(1, N + 1),
    "pickup_datetime": pickup_datetime,
    "pickup_community_area": np.random.choice(obszary, size=N, p=wagi_obszarow),
    "trip_miles": trip_miles,
    "trip_seconds": trip_seconds,
    "fare": fare,
    "tips": tips,
    "payment_type": payment_type,
    "company": company,
})

print(f"📊 Wygenerowano {len(taxi_df):,} przejazdów")
taxi_df.head()

## 🎓 Guided Example: Średnia stawka per godzina dnia

**Zadanie:** Sprawdźmy, jak zmienia się średnia stawka za przejazd w zależności od godziny dnia. Zróbmy to najpierw jako zapytanie BigQuery (Ścieżka A, referencyjnie), a potem — naprawdę — na naszym offline'owym DataFrame (Ścieżka B).

In [ ]:
# @title 🎓 Guided Example — Run and study this

# -- SQL (Ścieżka A) — referencyjnie, do uruchomienia na prawdziwym BigQuery:
# SELECT
#     EXTRACT(HOUR FROM trip_start_timestamp) AS godzina,
#     COUNT(*) AS liczba_przejazdow,
#     ROUND(AVG(fare), 2) AS srednia_stawka
# FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
# WHERE trip_start_timestamp BETWEEN '2023-01-01' AND '2023-01-31'
# GROUP BY godzina
# ORDER BY godzina;

# Ścieżka B — to samo pytanie, na offline'owym DataFrame:
taxi_df["godzina"] = taxi_df["pickup_datetime"].dt.hour

srednia_stawka_godzina = (
    taxi_df.groupby("godzina")
    .agg(liczba_przejazdow=("trip_id", "count"), srednia_stawka=("fare", "mean"))
    .round(2)
    .reset_index()
)

print(srednia_stawka_godzina)
srednia_stawka_godzina.plot(x="godzina", y="srednia_stawka", kind="bar", figsize=(10, 4), legend=False, title="Średnia stawka wg godziny dnia")

---
## ✏️ Ćwiczenie 1: Najbardziej ruchliwe dzielnice odbioru

**Trudność:** ⭐

Znajdź **5 dzielnic** (`pickup_community_area`) z największą liczbą przejazdów. Zapisz wynik jako `top5_dzielnic` — DataFrame z kolumnami `pickup_community_area` i `liczba_przejazdow`, posortowany malejąco po liczbie przejazdów.

> 💡 **Hint:** `groupby(...).size()` lub `groupby(...).agg(liczba_przejazdow=("trip_id", "count"))`, potem `.sort_values(...)` i `.head(5)`.

In [ ]:
# ✏️ Ćwiczenie 1: Najbardziej ruchliwe dzielnice odbioru

# TODO: policz liczbę przejazdów per pickup_community_area
top5_dzielnic = ...  # Zastąp ... swoim kodem

print(top5_dzielnic)

In [ ]:
# @title ✅ Test — Ćwiczenie 1 (uruchom, aby sprawdzić odpowiedź)

try:
    assert isinstance(top5_dzielnic, pd.DataFrame), "❌ Wynik powinien być DataFrame"
    assert len(top5_dzielnic) == 5, f"❌ Oczekiwano 5 wierszy, otrzymano {len(top5_dzielnic)}"
    assert "pickup_community_area" in top5_dzielnic.columns, "❌ Brakuje kolumny pickup_community_area"
    liczby = top5_dzielnic.iloc[:, -1].tolist()
    assert liczby == sorted(liczby, reverse=True), "❌ Wynik powinien być posortowany malejąco"
    assert 7 in top5_dzielnic["pickup_community_area"].values, "❌ Dzielnica 7 powinna być w TOP 5 (jest zdecydowanie popularniejsza)"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! Dasz radę 💪")

---
## ✏️ Ćwiczenie 2: Średni procent napiwku wg formy płatności

**Trudność:** ⭐⭐

Dla każdej wartości `payment_type` policz **średni procent napiwku** (`tips / fare * 100`). Zapisz wynik jako `sredni_procent_napiwku` — Series indeksowany po `payment_type`, wartości zaokrąglone do 2 miejsc po przecinku.

**Oczekiwany rezultat:** `Credit Card` i `Mobile` powinny mieć wyraźnie wyższy procent napiwku niż `Cash`.

> 💡 **Hint:** Dodaj kolumnę pomocniczą `procent_napiwku = taxi_df["tips"] / taxi_df["fare"] * 100`, potem `.groupby("payment_type")["procent_napiwku"].mean()`.

In [ ]:
# ✏️ Ćwiczenie 2: Średni procent napiwku wg formy płatności

# TODO: policz procent napiwku per przejazd, a potem średnią per payment_type
sredni_procent_napiwku = ...  # Zastąp ... swoim kodem

print(sredni_procent_napiwku)

In [ ]:
# @title ✅ Test — Ćwiczenie 2 (uruchom, aby sprawdzić odpowiedź)

try:
    assert isinstance(sredni_procent_napiwku, pd.Series), "❌ Wynik powinien być pd.Series"
    assert "Cash" in sredni_procent_napiwku.index, "❌ Brakuje 'Cash' w wyniku"
    assert "Credit Card" in sredni_procent_napiwku.index, "❌ Brakuje 'Credit Card' w wyniku"
    assert sredni_procent_napiwku["Credit Card"] > sredni_procent_napiwku["Cash"], "❌ Credit Card powinien mieć wyższy procent napiwku niż Cash"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! Dasz radę 💪")

---
## ✏️ Ćwiczenie 3: Korelacja dystansu i stawki

**Trudność:** ⭐⭐

Oblicz współczynnik korelacji Pearsona między `trip_miles` a `fare`. Zapisz wynik jako `korelacja_mile_stawka` (typ `float`).

> 💡 **Hint:** `.corr()` na dwóch kolumnach DataFrame lub `df["a"].corr(df["b"])`.

In [ ]:
# ✏️ Ćwiczenie 3: Korelacja dystansu i stawki

# TODO: policz korelację między trip_miles i fare
korelacja_mile_stawka = ...  # Zastąp ... swoim kodem

print(f"Korelacja trip_miles vs fare: {korelacja_mile_stawka:.3f}")

In [ ]:
# @title ✅ Test — Ćwiczenie 3 (uruchom, aby sprawdzić odpowiedź)

try:
    assert isinstance(korelacja_mile_stawka, float), "❌ Wynik powinien być liczbą float"
    assert 0.7 < korelacja_mile_stawka < 1.0, f"❌ Oczekiwano silnej dodatniej korelacji (0.7-1.0), otrzymano {korelacja_mile_stawka:.3f}"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! Dasz radę 💪")

---
## ✏️ Ćwiczenie 4: Dzień powszedni vs weekend

**Trudność:** ⭐⭐⭐

Sprawdź, czy średnia stawka (`fare`) różni się między dniami powszednimi a weekendem. Utwórz kolumnę pomocniczą `czy_weekend` (`True`/`False`, sobota i niedziela = weekend), a następnie policz średnią stawkę dla obu grup. Zapisz wynik jako `stawka_weekday_weekend` — Series indeksowany przez `czy_weekend`.

> 💡 **Hint:** `pickup_datetime.dt.dayofweek` zwraca 0 (poniedziałek) do 6 (niedziela); weekend to `>= 5`.

In [ ]:
# ✏️ Ćwiczenie 4: Dzień powszedni vs weekend

# TODO: utwórz kolumnę czy_weekend, a potem policz średnią stawkę per grupa
taxi_df["czy_weekend"] = ...  # Zastąp ... swoim kodem
stawka_weekday_weekend = ...  # Zastąp ... swoim kodem

print(stawka_weekday_weekend)

In [ ]:
# @title ✅ Test — Ćwiczenie 4 (uruchom, aby sprawdzić odpowiedź)

try:
    assert "czy_weekend" in taxi_df.columns, "❌ Brakuje kolumny czy_weekend"
    assert taxi_df["czy_weekend"].dtype == bool, "❌ czy_weekend powinno być typu bool"
    assert isinstance(stawka_weekday_weekend, pd.Series), "❌ Wynik powinien być pd.Series"
    assert set(stawka_weekday_weekend.index) == {True, False}, "❌ Indeks powinien zawierać True i False"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! Dasz radę 💪")

---
## ✏️ Ćwiczenie 5: Rozkład form płatności

**Trudność:** ⭐⭐

Policz **procentowy udział** każdej formy płatności (`payment_type`) w całym zbiorze danych. Zapisz wynik jako `rozklad_platnosci` — Series z wartościami procentowymi (0-100), zaokrąglonymi do 1 miejsca po przecinku, posortowany malejąco.

> 💡 **Hint:** `.value_counts(normalize=True) * 100`.

In [ ]:
# ✏️ Ćwiczenie 5: Rozkład form płatności

# TODO: policz procentowy udział payment_type
rozklad_platnosci = ...  # Zastąp ... swoim kodem

print(rozklad_platnosci)

In [ ]:
# @title ✅ Test — Ćwiczenie 5 (uruchom, aby sprawdzić odpowiedź)

try:
    assert isinstance(rozklad_platnosci, pd.Series), "❌ Wynik powinien być pd.Series"
    assert abs(rozklad_platnosci.sum() - 100) < 0.5, f"❌ Suma procentów powinna wynosić ~100, otrzymano {rozklad_platnosci.sum():.1f}"
    assert rozklad_platnosci.index[0] == "Credit Card", "❌ Credit Card powinna być najczęstszą formą płatności"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! Dasz radę 💪")

---
## ✏️ Ćwiczenie 6: Ile to kosztuje? 💰 Szacowanie kosztu zapytania BigQuery

**Trudność:** ⭐⭐⭐⭐

Wyobraź sobie, że tabela `bigquery-public-data.chicago_taxi_trips.taxi_trips` ma **200 000 000** wierszy. Poniżej znajdziesz uproszczoną tabelę kosztów pamięciowych per kolumna (przybliżone średnie bajty/wiersz w kolumnowym storage BigQuery) — to typowe wartości dla typów `INT64`/`FLOAT64`/`TIMESTAMP` (8 bajtów) i `STRING` (zmienna długość).

Napisz zapytanie SQL (jako string w Pythonie, referencyjnie) wybierające kolumny: `trip_start_timestamp`, `trip_miles`, `fare`, `payment_type`. Oblicz, ile **GB** to zapytanie zeskanuje oraz ile będzie kosztować w modelu **on-demand** ($7.50 / TB).

Zapisz wyniki jako `bajty_na_wiersz_lacznie` (int/float), `gb_do_zeskanowania` (float) i `koszt_usd` (float).

In [ ]:
# ✏️ Ćwiczenie 6: Szacowanie kosztu zapytania BigQuery

LICZBA_WIERSZY = 200_000_000
CENA_ZA_TB_USD = 7.50

# Przybliżone bajty/wiersz per kolumna (kolumnowy storage BigQuery)
BAJTY_NA_KOLUMNE = {
    "trip_start_timestamp": 8,
    "trip_end_timestamp": 8,
    "trip_seconds": 8,
    "trip_miles": 8,
    "fare": 8,
    "tips": 8,
    "trip_total": 8,
    "payment_type": 20,
    "company": 25,
    "pickup_community_area": 8,
    "dropoff_community_area": 8,
}

wybrane_kolumny = ["trip_start_timestamp", "trip_miles", "fare", "payment_type"]

# TODO 1: zsumuj bajty/wiersz dla wybranych kolumn
bajty_na_wiersz_lacznie = ...  # Zastąp ... swoim kodem

# TODO 2: pomnóż przez liczbę wierszy i przelicz na GB (1 GB = 1e9 bajtów)
gb_do_zeskanowania = ...  # Zastąp ... swoim kodem

# TODO 3: przelicz koszt w USD (1 TB = 1e12 bajtów, cena za TB podana wyżej)
koszt_usd = ...  # Zastąp ... swoim kodem

print(f"Zapytanie zeskanuje ~{gb_do_zeskanowania:.2f} GB, koszt: ~${koszt_usd:.4f}")

In [ ]:
# @title ✅ Test — Ćwiczenie 6 (uruchom, aby sprawdzić odpowiedź)

try:
    assert bajty_na_wiersz_lacznie == 44, f"❌ Oczekiwano 44 bajty/wiersz (8+8+8+20), otrzymano {bajty_na_wiersz_lacznie}"
    oczekiwane_gb = (bajty_na_wiersz_lacznie * LICZBA_WIERSZY) / 1e9
    assert abs(gb_do_zeskanowania - oczekiwane_gb) < 0.01, f"❌ Oczekiwano ~{oczekiwane_gb:.2f} GB, otrzymano {gb_do_zeskanowania:.2f}"
    oczekiwany_koszt = (bajty_na_wiersz_lacznie * LICZBA_WIERSZY) / 1e12 * CENA_ZA_TB_USD
    assert abs(koszt_usd - oczekiwany_koszt) < 0.001, f"❌ Oczekiwano ~${oczekiwany_koszt:.4f}, otrzymano ${koszt_usd:.4f}"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
    print(f"💡 Dla porównania: SELECT * (wszystkie 11 kolumn) kosztowałoby znacznie więcej!")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! Dasz radę 💪")

---
## 🏆 Bonus Challenge (opcjonalnie): PySpark na Dataproc

Wyobraź sobie, że zamiast pandas na małym DataFrame, masz **pełne 200 mln wierszy** i uruchamiasz job na klastrze Dataproc. Przepisz logikę z Ćwiczenia 1 (najbardziej ruchliwe dzielnice) na PySpark. Kod poniżej nie jest testowany automatycznie — potraktuj to jako ćwiczenie koncepcyjne (nie wymaga działającego klastra Spark).

In [ ]:
# 🏆 Bonus: równoważny kod PySpark (jak na klastrze Dataproc)
# Ten kod jest przykładem koncepcyjnym — nie uruchamiamy go w tym notebooku.

kod_pyspark = '''
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("TopDzielnice").getOrCreate()

# Odczyt bezpośrednio z BigQuery (spark-bigquery-connector) na klastrze Dataproc
df = (spark.read
      .format("bigquery")
      .option("table", "bigquery-public-data.chicago_taxi_trips.taxi_trips")
      .load())

top5_dzielnic_spark = (
    df.groupBy("pickup_community_area")
      .agg(F.count("*").alias("liczba_przejazdow"))
      .orderBy(F.desc("liczba_przejazdow"))
      .limit(5)
)
top5_dzielnic_spark.show()
spark.stop()
'''

print(kod_pyspark)
print("💡 Ten sam wynik co Ćwiczenie 1, ale skalowalny do 200 mln+ wierszy na klastrze Dataproc!")

---
## 📝 Podsumowanie

Świetna robota! 🎉 W tym ćwiczeniu przećwiczyłeś/aś:

- ✅ Analizę danych o przejazdach taksówek (godziny szczytu, dzielnice, napiwki, dystanse)
- ✅ Pracę z dwoma ścieżkami dostępu do danych — prawdziwym BigQuery i offline'owym fallbackiem
- ✅ Szacowanie **kosztu** zapytania BigQuery na podstawie przeskanowanych bajtów
- ✅ Przekładanie logiki SQL/pandas na PySpark, jak na klastrze Dataproc

### 📚 Dalsza lektura
- [BigQuery public datasets](https://cloud.google.com/bigquery/public-data)
- [Understanding BigQuery pricing](https://cloud.google.com/bigquery/pricing)
- [Dataproc Documentation](https://cloud.google.com/dataproc/docs)

### 🔮 Co dalej?
❄️ **Przerwa świąteczna!** Nie ma zajęć 22.12.2026 ani 29.12.2026. Do zobaczenia na **Zajęciach nr 12 (12.01.2027)** — **Przetwarzanie strumieniowe: Structured Streaming i Kafka**!